In [5]:
import numpy as np
import pandas as pd
from scipy.special import roots_hermitenorm

def sinh(z, a):
    if np.isclose(a, 0.0):
        return z
    return np.sinh(a * z) / a


def tanh(z, a):
    return z + a * np.tanh(z)

def cubic(z, strength):
    return z + strength * z**3

def exp(z, strength):
    if strength == 0:
        return z
    return np.expm1(strength * z) / strength

def standardize_transform(transform, strength, n_quad=200):
    x,w = roots_hermitenorm(n_quad)
    w = w / np.sqrt(2 * np.pi)

    y = transform(x, strength)

    mean = np.sum(w*y)
    var = np.sum(w * (y-mean)**2)
    sd = np.sqrt(var)
    def standardized(z):
        return (transform(z, strength) - mean) / sd
    
    return standardized

def curvature(f, z, h=1e-4):
    z = np.asarray(z, dtype=float)

    f_plus = f(z+h)
    f_zero = f(z)
    f_minus = f(z-h)

    first = (f_plus-f_minus)/(2*h)
    second = (f_plus - 2*f_zero + f_minus)/ h**2
    kappa= np.abs(second)/(1+first**2)**1.5

    return kappa

def mean_curvature(transform, strength, n_quad=200):
    x, w = roots_hermitenorm(n_quad)
    w = w / np.sqrt(2 * np.pi)

    f = standardize_transform(transform, strength, n_quad=n_quad)

    kappa = curvature(f,x)
    return np.sum(w*kappa)

In [6]:
TRANSFORMS = {
    "cubic": cubic,
    "sinh": sinh,
    "exp": exp,
    "tanh": tanh,
}


STRENGTHS = {
    "cubic": [0.0, 0.10, 0.25, 0.50, 1.00],
    "sinh":  [0.0, 0.35, 0.70, 1.00, 1.30],
    "exp":   [0.0, 0.20, 0.40, 0.60, 0.80],
    "tanh":  [0.0, 0.25, 0.50, 1.00, 2.00],
}


curvature_rows = []


for transform_name, strengths in STRENGTHS.items():
    for strength in strengths:

        curvature_val = mean_curvature(
            TRANSFORMS[transform_name],
            strength
        )

        row = {
            "transform": transform_name,
            "strength": strength
        }
        row["curvature"] = curvature_val
        
        curvature_rows.append(row)


curvature_results = pd.DataFrame(curvature_rows)

print(curvature_results)

curvature_results.to_csv(
    "../results/curvature_metrics.csv",
    index=False
)

   transform  strength  curvature
0      cubic      0.00   0.000000
1      cubic      0.10   0.110823
2      cubic      0.25   0.207136
3      cubic      0.50   0.300125
4      cubic      1.00   0.384560
5       sinh      0.00   0.000000
6       sinh      0.35   0.031321
7       sinh      0.70   0.103360
8       sinh      1.00   0.182261
9       sinh      1.30   0.260236
10       exp      0.00   0.000000
11       exp      0.20   0.069985
12       exp      0.40   0.135348
13       exp      0.60   0.191117
14       exp      0.80   0.233150
15      tanh      0.00   0.000000
16      tanh      0.25   0.038219
17      tanh      0.50   0.067685
18      tanh      1.00   0.110905
19      tanh      2.00   0.164345


In [7]:
def quintic(z,a):
    z = np.asarray(z, dtype=float)
    return z + a * z**5

def quad_cubic(z, a):
    z = np.asarray(z, dtype=float)
    return z + a * z**2 + (a**2 / 3.0) * z**3

TRANSFORMS = {
    "cubic": cubic,
    "quintic": quintic,
    "exp": exp,
    "quad": quad_cubic,
}

STRENGTHS = {
    "cubic": [0.1302579, 0.22996560, 0.3532381, 0.5265986, 1.3483315],
    "quintic":  [0.0098076, 0.0152675, 0.0204604, 0.0258628, 0.038476],
    "exp":   [0.3189425, 0.4551335, 0.5627497, 0.6563857, 0.8218707],
    "quad":  [0.1659280, 0.2476384, 0.3223790, 0.3994894, 0.5912237],
}

curvature_rows = []


for transform_name, strengths in STRENGTHS.items():
    for strength in strengths:

        curvature_val = mean_curvature(
            TRANSFORMS[transform_name],
            strength
        )

        row = {
            "transform": transform_name,
            "strength": strength
        }
        row["curvature"] = curvature_val
        
        curvature_rows.append(row)


curvature_results = pd.DataFrame(curvature_rows)

print(curvature_results)

curvature_results.to_csv(
    "../results/matched_curvature_metrics.csv",
    index=False
)


   transform  strength  curvature
0      cubic  0.130258   0.134204
1      cubic  0.229966   0.196668
2      cubic  0.353238   0.252825
3      cubic  0.526599   0.307069
4      cubic  1.348332   0.413202
5    quintic  0.009808   0.058165
6    quintic  0.015267   0.078901
7    quintic  0.020460   0.096266
8    quintic  0.025863   0.112551
9    quintic  0.038476   0.144765
10       exp  0.318943   0.109737
11       exp  0.455134   0.151882
12       exp  0.562750   0.181667
13       exp  0.656386   0.204499
14       exp  0.821871   0.236791
15      quad  0.165928   0.118046
16      quad  0.247638   0.172053
17      quad  0.322379   0.213697
18      quad  0.399489   0.247565
19      quad  0.591224   0.302880
